In [10]:
!pip install datasets pyarrow "selectolax<1.0" py-data-juicer
from huggingface_hub import login
login()

In [11]:
###Loading data from CulturaX, we can link to any other datasets.

import os
import pandas as pd
from datasets import load_dataset

print("Streaming CulturaX (sv) chunk...")
# The streaming=True flag prevents downloading the entire dataset
dataset = load_dataset("uonlp/CulturaX", "sv", split="train", streaming=True)

# Pull exactly 50,000 documents for the scaled test
raw_df = pd.DataFrame(dataset.take(10000))
raw_df["doc_id"] = raw_df.index.astype(str)
os.makedirs("culturax_sv_full_folder", exist_ok=True)

# Convert to Pandas and save as a partitioned Parquet file
#df = pd.DataFrame(chunk)
# Save as a compressed Parquet file inside the folder
raw_df.to_parquet("culturax_sv_full_folder/chunk_0.parquet", compression="snappy")
print(f"Saved {len(raw_df)} documents to the partitioned directory.")

Streaming CulturaX (sv) chunk...


Resolving data files:   0%|          | 0/64 [00:00<?, ?it/s]

Saved 10000 documents to the partitioned directory.


#### The cell below contains som mini examples of a uncleaned textda, just to check if the pipeline works.

In [22]:
# import pandas as pd
# import os

# test_data = [
#     {
#         "doc_id": "test_01_valid",
#         "url": "https://test.se/valid",
#         # Helt vanlig, välformulerad svensk text. Ska passera alla filter helt orörd.
#         "text": "Artificiell intelligens och maskininlärning har under det senaste årtiondet genomgått en enorm utveckling. Från att tidigare ha varit ett smalt forskningsområde, är teknologin nu en integrerad del av vår vardag. Vi ser det i allt från röstassistenter i våra mobiltelefoner till avancerade algoritmer som styr självkörande bilar och medicinsk diagnostik. Denna utveckling drivs framförallt av tillgången till enorma mängder data och kraftigare processorer som kan träna djupa neurala nätverk. Samtidigt väcker tekniken viktiga etiska frågor kring integritet, arbetsmarknadens förändring och hur vi säkerställer att AI-system fattar rättvisa och transparenta beslut i samhället."
#     },
#     {
#         "doc_id": "test_02_english",
#         "url": "https://test.se/eng",
#         # Fullång engelsk text. Ska fällas av language_id_score_filter.
#         "text": "Artificial intelligence and machine learning have undergone tremendous development over the last decade. From previously being a narrow area of research, the technology is now an integrated part of our everyday lives. We see it in everything from voice assistants in our mobile phones to advanced algorithms that control self-driving cars and medical diagnostics. This development is primarily driven by the availability of massive amounts of data and more powerful processors capable of training deep neural networks. At the same time, the technology raises important ethical questions regarding privacy and automation."
#     },
#     {
#         "doc_id": "test_03_cookie_links_metadata",
#         "url": "https://test.se/cookie",
#         # Giltig svensk brödtext, men inbakad mellan cookie-banners, länkar och blogg-metadata.
#         # Mappers ska städa bort första raden och de sista raderna, men bevara mitten.
#         "text": "Vi använder cookies för att förbättra din upplevelse. Läs mer om cookies här.\nArtificiell intelligens och maskininlärning har under det senaste årtiondet genomgått en enorm utveckling. Från att tidigare ha varit ett smalt forskningsområde, är teknologin nu en integrerad del av vår vardag. Vi ser det i allt från röstassistenter i våra mobiltelefoner till avancerade algoritmer som styr självkörande bilar och medicinsk diagnostik.\nLäs mer på vår hemsida om detta spännande ämne.\nav Fredrik Edin 24 november, 2010 13:02\nTaggad med: AI, teknik, framtid\nTweets that mention den här artikeln."
#     },
#     {
#         "doc_id": "test_04_short_lines_menu",
#         "url": "https://test.se/menu",
#         # Lång text, men består enbart av korta rader (navigering/meny).
#         # Ska fällas av average_line_length_filter.
#         "text": "Startsida\nNyheter och press\nSport och fritid\nEkonomi och finans\nKontakta vår kundtjänst\nOm företaget och vår historia\nJobba hos oss på kontoret\nLäs vår integritetspolicy\nLogga in på dina sidor\nSkapa ett nytt användarkonto\nSök efter gamla artiklar\nPrenumerera på nyhetsbrevet\nFölj oss på sociala medier\nVanliga frågor och svar\nAllmänna köpvillkor\nInformation om leveranser\nReturer och reklamationer\nLediga tjänster just nu\nVår styrelse och ledning\nInvesterarrelationer och pressmeddelanden"
#     },
#     {
#         "doc_id": "test_05_garbage_chars",
#         "url": "https://test.se/garbage",
#         # Fullång text men proppfull med trasig HTML och specialtecken.
#         # Ska fällas av alphanumeric_filter eller special_characters_filter.
#         "text": "Detta är en testtext för att se om filtret fungerar. Men här kommer problemen: %%% ### @@@ ||| \\u003e \\u003c <html><body><div>Trasig kod i mängder</div></body></html> &amp; &lt; &gt; %%% ### @@@ ||| \\u003e \\u003c <html><body><div>Trasig kod i mängder</div></body></html> &amp; &lt; &gt; %%% ### @@@ ||| \\u003e \\u003c <html><body><div>Trasig kod i mängder</div></body></html> &amp; &lt; &gt; %%% ### @@@ ||| \\u003e \\u003c <html><body><div>Trasig kod i mängder</div></body></html> &amp; &lt; &gt; Denna text saknar tillräckligt många vanliga bokstäver."
#     },
#     {
#         "doc_id": "test_06_near_duplicate",
#         "url": "https://test.se/dup",
#         # Identisk med test_01 i början, men sista meningen är utbytt.
#         # Ska fällas av document_simhash_deduplicator (utan att behöva multipliceras).
#         "text": "Artificiell intelligens och maskininlärning har under det senaste årtiondet genomgått en enorm utveckling. Från att tidigare ha varit ett smalt forskningsområde, är teknologin nu en integrerad del av vår vardag. Vi ser det i allt från röstassistenter i våra mobiltelefoner till avancerade algoritmer som styr självkörande bilar och medicinsk diagnostik. Denna utveckling drivs framförallt av tillgången till enorma mängder data och kraftigare processorer som kan träna djupa neurala nätverk. Samtidigt väcker tekniken intressanta filosofiska tankar kring hur vi bygger morgondagens smarta städer och infrastruktur."
#     }
# ]

# # Skapa testkatalogen och spara Parquet-filen
# os.makedirs("test_data_folder", exist_ok=True)
# pd.DataFrame(test_data).to_parquet("test_data_folder/test_chunk.parquet", compression="snappy")
# print("Robusta testfall sparade i 'test_data_folder/test_chunk.parquet'!")

Robusta testfall sparade i 'test_data_folder/test_chunk.parquet'!


There Two inbuilt functions in the Data-Juicer that we can use: SimHash or MinHash. when i have tested on test cases above MinHash seems to be more
efficient when it comes the removing duplicates. So i theink we go for MinHash.


In [41]:
###This is the config file for SimHash

# %%writefile config.yaml
# # dataset_path: "culturax_sv_full_folder"
# # export_path: "culturax_sv_cleaned.parquet"

# dataset_path: "test_data_folder"
# export_path: "test_data_cleaned.parquet"
# np: 2
# # ... (behåll alla dina mappers och filter exakt som de är) ...
# np: 2
# skip_op_error: false   # fail loudly instead of silently skipping a broken op

# process:
#   #MAPPERS FIRST: clean the text, then judge it
#   - clean_html_mapper: {}
#   - clean_links_mapper: {}
#   - clean_email_mapper: {}
#   - fix_unicode_mapper: {}
#   - whitespace_normalization_mapper: {}

#   # Blog / CMS boilerplate lines (tags, categories, share buttons)
#   - replace_content_mapper:
#       pattern: '(?mi)^\s*(taggad med|taggar|etiketter|kategori(er)?|arkiverad under|publicerad i|dela( på| detta)?|share this|gilla detta)\b[^\n]*$'
#       repl: ''
#   # Twitter / pingback / trackback lines
#   - replace_content_mapper:
#       pattern: '(?mi)^[^\n]*(tweets that mention|this post was mentioned on twitter|pingback|trackback)[^\n]*$'
#       repl: ''
#   # "[…] excerpt" lines
#   - replace_content_mapper:
#       pattern: '(?m)^[ \t]*\[(…|\.\.\.)\][^\n]*$'
#       repl: ''
#   # Comment headers: "Anonym  den 24 november, 2010 kl. 15:56"
#   - replace_content_mapper:
#       pattern: '(?mi)^[^\n]{0,80}?\bden \d{1,2} (januari|februari|mars|april|maj|juni|juli|augusti|september|oktober|november|december),? \d{4}( kl\.? ?\d{1,2}[:.]\d{2})?[^\n]{0,20}$'
#       repl: ''
#   # Bylines: "av Fredrik Edin 24 november, 2010 13:02"
#   - replace_content_mapper:
#       pattern: '(?mi)^\s*av [^\n]{1,60}?\d{1,2} (januari|februari|mars|april|maj|juni|juli|augusti|september|oktober|november|december),? ?\d{4}[^\n]*$'
#       repl: ''
#   # Cookie banners
#   - replace_content_mapper:
#       pattern: '(?mi)^[^\n]*(använder (cookies|kakor)|godkänn cookies|läs mer om cookies)[^\n]*$'
#       repl: ''
#   # Teaser / "read more" lines
#   - replace_content_mapper:
#       pattern: '(?mi)^\s*(läs mer|läs hela|fortsätt läsa|fortsättning följer)[^\n]{0,40}$'
#       repl: ''
#   # Collapse leftover spaces and blank lines
#   - replace_content_mapper:
#       pattern: '[ \t]{2,}'
#       repl: ' '
#   - replace_content_mapper:
#       pattern: '\n[ \t]*\n+'
#       repl: '\n'
#   - remove_repeat_sentences_mapper: {}

#   # QUALITY FILTERS on the cleaned text
#   # (all model-free except language ID, which downloads fine)
#   - text_length_filter:
#       min_len: 200
#       max_len: 8000
#   - average_line_length_filter:
#       min_len: 30
#   - alphanumeric_filter:
#       tokenization: false
#       min_ratio: 0.70
#   - special_characters_filter:
#       max_ratio: 0.25
#   - character_repetition_filter:
#       rep_len: 10
#       max_ratio: 0.15
#   - word_repetition_filter:
#       tokenization: false
#       rep_len: 10
#       max_ratio: 0.15
#   - words_num_filter:
#       tokenization: false
#       min_num: 40
#   - language_id_score_filter:
#       lang: "sv"
#       min_score: 0.90

#   # perplexity_filter REMOVED: its Swedish KenLM/sentencepiece model
#   # (sv.sp.model) can no longer be downloaded (403 + 404), which crashed the run.

#   # ---------- 3. NEAR-DUPLICATE REMOVAL ----------
#   - document_simhash_deduplicator:
#       tokenization: space
#       hamming_distance: 8
#       num_blocks: 9


Overwriting config.yaml


In [20]:
# ### The config file for MinHash, i only has run it on test cases not any big dataset yet, which i will do


# %%writefile config.yaml
# ###test mode
# # dataset_path: "test_data_folder"
# # export_path: "test_data_cleaned.parquet"

# ## real data
# dataset_path: "culturax_sv_full_folder"
# export_path: "culturax_sv_cleaned.parquet"
# np: 2
# skip_op_error: false

# process:
#   # IN-PLACE MAPPERS
#   - clean_html_mapper: {}
#   - clean_links_mapper: {}
#   - clean_email_mapper: {}
#   - fix_unicode_mapper: {}
#   - whitespace_normalization_mapper: {}

#   - replace_content_mapper:
#       pattern: '(?mi)^\s*(taggad med|taggar|etiketter|kategori(er)?|arkiverad under|publicerad i|dela( på| detta)?|share this|gilla detta)\b[^\n]*$'
#       repl: ''
#   - replace_content_mapper:
#       pattern: '(?mi)^[^\n]*(tweets that mention|this post was mentioned on twitter|pingback|trackback)[^\n]*$'
#       repl: ''
#   - replace_content_mapper:
#       pattern: '(?m)^[ \t]*\[(…|\.\.\.)\][^\n]*$'
#       repl: ''
#   - replace_content_mapper:
#       pattern: '(?mi)^[^\n]{0,80}?\bden \d{1,2} (januari|februari|mars|april|maj|juni|juli|augusti|september|oktober|november|december),? \d{4}( kl\.? ?\d{1,2}[:.]\d{2})?[^\n]{0,20}$'
#       repl: ''
#   - replace_content_mapper:
#       pattern: '(?mi)^\s*av [^\n]{1,60}?\d{1,2} (januari|februari|mars|april|maj|juni|juli|augusti|september|oktober|november|december),? ?\d{4}[^\n]*$'
#       repl: ''
#   - replace_content_mapper:
#       pattern: '(?mi)^[^\n]*(använder (cookies|kakor)|godkänn cookies|läs mer om cookies)[^\n]*$'
#       repl: ''
#   - replace_content_mapper:
#       pattern: '(?mi)^\s*(läs mer|läs hela|fortsätt läsa|fortsättning följer)[^\n]{0,40}$'
#       repl: ''
#   - replace_content_mapper:
#       pattern: '(?i)Warning:\s+[^\n]+?on line\s+\d+'
#       repl: ''
#   - replace_content_mapper:
#       pattern: '(?m)^[←→\<\>]\s*.*$|^.*\s*[←→\<\>]$'
#       repl: ''
#   - replace_content_mapper:
#       pattern: '(?mi)^(Postat|Publicerat) (den )?\d{4}-\d{1,2}-\d{1,2}.*$'
#       repl: ''
#   - replace_content_mapper:
#       pattern: '(?mi)^Läs även andra bloggares åsikter.*$'
#       repl: ''

#   - whitespace_normalization_mapper: {} ##
#   - replace_content_mapper:
#       pattern: '[ \t]{2,}'
#       repl: ' '
#   - replace_content_mapper:
#       pattern: '\n[ \t]*\n+'
#       repl: '\n'
#   - remove_repeat_sentences_mapper: {}

#   # QUALITY FILTERS (Filtrering)
#   - text_length_filter:
#       min_len: 200
#       max_len: 8000
#   - average_line_length_filter:
#       min_len: 30

#   - alphanumeric_filter:
#       tokenization: false
#       min_ratio: 0.72  ##

#   - special_characters_filter:
#       max_ratio: 0.22   ##25
#   - character_repetition_filter:
#       rep_len: 10
#       max_ratio: 0.15
#   - word_repetition_filter:
#       tokenization: false
#       rep_len: 10
#       max_ratio: 0.15
#   - words_num_filter:
#       tokenization: false
#       min_num: 40
#   - language_id_score_filter:
#       lang: "sv"
#       min_score: 0.90

#   - perplexity_filter:  ##
#       lang: "sv"
#       max_ppl: 1500

#   # MINHASH DEDUPLICATION
#   - document_minhash_deduplicator:
#       tokenization: space
#       num_permutations: 256
#       jaccard_threshold: 0.65    ######## OBSSSS   ##### Standard is around 0.65, 0.70 0.8. But for the test cases above 0.4 works perfect


%%writefile config.yaml
dataset_path: "culturax_sv_full_folder"
export_path: "culturax_sv_cleaned.parquet"
np: 2
skip_op_error: false

process:
  # ---------- 1. IN-PLACE MAPPERS (Städning) ----------
  - clean_html_mapper: {}
  - clean_links_mapper: {}
  - clean_email_mapper: {}
  - fix_unicode_mapper: {}

  # Navigeringsstigar / Breadcrumbs
  - replace_content_mapper:
      pattern: '(?m)^([^\n]*>){2,}[^\n]*$'
      repl: ''
  - replace_content_mapper:
      pattern: '(?mi)^\s*(You are here|Hem|Home|Start)\s*[:/].*$'
      repl: ''

  # Publiceringsdatum
  - replace_content_mapper:
      pattern: '(?mi)^\s*(Publicerat|Uppdaterad|Skapad|Postat) [^\n]*$'
      repl: ''

  # Produktmetadata
  - replace_content_mapper:
      pattern: '(?mi)^\s*(ISBN|EAN|Art nr|Artikelnummer)[^\n]*$'
      repl: ''

  # Sidfötter och UI-skräp
  - replace_content_mapper:
      pattern: '(?m)^[\d\s\+\-\:\;\<\>]{15,}$'
      repl: ''

  # Blogg/CMS-rensning
  - replace_content_mapper:
      pattern: '(?mi)^\s*(taggad med|taggar|etiketter|kategori(er)?|arkiverad under|publicerad i|dela( på| detta)?|share this|gilla detta)\b[^\n]*$'
      repl: ''
  - replace_content_mapper:
      pattern: '(?mi)^[^\n]*(tweets that mention|this post was mentioned on twitter|pingback|trackback)[^\n]*$'
      repl: ''
  - replace_content_mapper:
      pattern: '(?m)^[ \t]*\[(…|\.\.\.)\][^\n]*$'
      repl: ''
  - replace_content_mapper:
      pattern: '(?mi)^[^\n]{0,80}?\bden \d{1,2} (januari|februari|mars|april|maj|juni|juli|augusti|september|oktober|november|december),? \d{4}( kl\.? ?\d{1,2}[:.]\d{2})?[^\n]{0,20}$'
      repl: ''
  - replace_content_mapper:
      pattern: '(?mi)^\s*av [^\n]{1,60}?\d{1,2} (januari|februari|mars|april|maj|juni|juli|augusti|september|oktober|november|december),? ?\d{4}[^\n]*$'
      repl: ''
  - replace_content_mapper:
      pattern: '(?mi)^[^\n]*(använder (cookies|kakor)|godkänn cookies|läs mer om cookies)[^\n]*$'
      repl: ''
  - replace_content_mapper:
      pattern: '(?mi)^\s*(läs mer|läs hela|fortsätt läsa|fortsättning följer)[^\n]{0,40}$'
      repl: ''

  # Normaliserar blanksteg
  - whitespace_normalization_mapper: {}
  - replace_content_mapper:
      pattern: '[ \t]{2,}'
      repl: ' '
  - replace_content_mapper:
      pattern: '\n[ \t]*\n+'
      repl: '\n'
  - remove_repeat_sentences_mapper: {}

  # ---------- 2. QUALITY FILTERS (Filtrering) ----------
  - text_length_filter:
      min_len: 200
      max_len: 8000
  - average_line_length_filter:
      min_len: 30

  - alphanumeric_filter:
      tokenization: false
      min_ratio: 0.72

  - special_characters_filter:
      max_ratio: 0.22
  - character_repetition_filter:
      rep_len: 10
      max_ratio: 0.15
  - word_repetition_filter:
      tokenization: false
      rep_len: 10
      max_ratio: 0.15
  - words_num_filter:
      tokenization: false
      min_num: 40
  - language_id_score_filter:
      lang: "sv"
      min_score: 0.90

  # ---------- 3. MINHASH DEDUPLICATION ----------
  - document_minhash_deduplicator:
      tokenization: space
      num_permutations: 256
      jaccard_threshold: 0.65

Overwriting config.yaml


In [40]:
### This just runs for the example cases and need to be modified if testing bigger datasets

# import os, subprocess, pandas as pd
# import difflib

# # 1. Radera gammal testdata för säkerhets skull
# if os.path.exists("test_data_cleaned.parquet"):
#     os.remove("test_data_cleaned.parquet")

# # 2. Kör Data-Juicer
# print("Kör Data-Juicer med MinHash...")
# res = subprocess.run(["dj-process", "--config", "config.yaml"], capture_output=True, text=True)
# if res.returncode != 0 or not os.path.exists("test_data_cleaned.parquet"):
#     print(res.stderr[-2000:])
#     raise RuntimeError("Data-Juicer kraschade.")

# # 3. Ladda testresultaten
# raw_df = pd.read_parquet("test_data_folder/test_chunk.parquet")
# clean_df = pd.read_parquet("test_data_cleaned.parquet")

# print("=" * 50)
# print(f"Testdokument från början: {len(raw_df)}")
# print(f"Testdokument som överlevde: {len(clean_df)}")
# print("=" * 50)

# # 4. Kontrollera om MinHash kastade inneslutningen (test_03)
# dropped = raw_df[~raw_df['doc_id'].isin(clean_df['doc_id'])]
# print("\n❌ DESSA DOKUMENT KASTADES AV FILTREN ELLER MINHASH:")
# display(dropped[['doc_id']])

Kör Data-Juicer med MinHash...
Testdokument från början: 6
Testdokument som överlevde: 1

❌ DESSA DOKUMENT KASTADES AV FILTREN ELLER MINHASH:


,doc_id
1,test_02_english
2,test_03_cookie_links_metadata
3,test_04_short_lines_menu
4,test_05_garbage_chars
5,test_06_near_duplicate


In [29]:
# 1. Installera Data-Juicer och beroenden igen (eftersom sessionen startats om)


# !pip install py-data-juicer datasets pyarrow "selectolax<1.0"

# import os, subprocess, pandas as pd
# import difflib

# # 2. Radera gammal testdata
# if os.path.exists("test_data_cleaned.parquet"):
#     os.remove("test_data_cleaned.parquet")

# # 3. Kör Data-Juicer
# print("Kör Data-Juicer på testdatan...")
# !dj-process --config config.yaml

# # 4. Ladda testresultaten
# raw_df = pd.read_parquet("test_data_folder/test_chunk.parquet")
# clean_df = pd.read_parquet("test_data_cleaned.parquet")

# print("=" * 50)
# print(f"Testdokument från början: {len(raw_df)}")
# print(f"Testdokument som överlevde: {len(clean_df)}")
# print("=" * 50)

# # 5. Visa vilka som raderades helt
# dropped = raw_df[~raw_df['doc_id'].isin(clean_df['doc_id'])]
# print("\n❌ DESSA DOKUMENT KASTADES AV FILTREN:")
# display(dropped[['doc_id']])

# # 6. Visa städningen i de som överlevde
# print("\n✅ DESSA ÖVERLEVDE (Och så här städades de):")
# merged = pd.merge(
#     raw_df[["doc_id", "text"]].rename(columns={"text": "raw_text"}),
#     clean_df[["doc_id", "text"]].rename(columns={"text": "clean_text"}),
#     on="doc_id"
# )

# for _, row in merged.iterrows():
#     print(f"\n--- {row['doc_id']} ---")
#     if row["raw_text"] != row["clean_text"]:
#         print("ÄNDRINGAR (Borttaget skräp):")
#         diff = difflib.ndiff(row["raw_text"].splitlines(keepends=True),
#                              row["clean_text"].splitlines(keepends=True))
#         for line in diff:
#             if line.startswith("- "):
#                 print(line.strip())
#     else:
#         print("Inga ändringar gjordes i texten (perfekt baslinje).")

Kör Data-Juicer på testdatan...
2026-10-13 23:45:20.893 | INFO     | data_juicer.config.config:1020 - dataset_path config is set and a valid local path
2026-10-13 23:45:21.407 | INFO     | data_juicer.config.config:920 - 🔍 Checking for job resumption: 20261013_234520_5c0078
2026-10-13 23:45:21.408 | WARNING  | data_juicer.config.config:1426 - No config file found in work directory: /content/20261013_234520_5c0078
2026-10-13 23:45:21.408 | INFO     | data_juicer.config.config:1667 - Back up the input config file [/content/config.yaml] to [/content/20261013_234520_5c0078/config.yaml]
2026-10-13 23:45:21.409 | INFO     | data_juicer.config.config:1408 - 💾 Saved CLI arguments to: /content/20261013_234520_5c0078/cli.yaml
2026-10-13 23:45:21.426 | INFO     | data_juicer.config.config:1692 - Configuration table: 
╒═════════════════════════════════════╤══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════╕
│ key  

,doc_id
1,test_02_english
3,test_04_short_lines_menu
4,test_05_garbage_chars
5,test_06_near_duplicate



✅ DESSA ÖVERLEVDE (Och så här städades de):

--- test_01_valid ---
Inga ändringar gjordes i texten (perfekt baslinje).

--- test_03_cookie_links_metadata ---
ÄNDRINGAR (Borttaget skräp):
- Vi använder cookies för att förbättra din upplevelse. Läs mer om cookies här.
- Läs mer på vår hemsida om detta spännande ämne.
- av Fredrik Edin 24 november, 2010 13:02
- Taggad med: AI, teknik, framtid
- Tweets that mention den här artikeln.


In [44]:
### .parquet -> readable text (test cases)

# import pandas as pd

# # Läs in den binära parquet-filen till en Pandas DataFrame
# df = pd.read_parquet("test_data_cleaned.parquet")

# # 1. Visa en översikt av filen (som en tabell)
# print("ÖVERSIKT AV DATAN:")
# display(df)

# print("\n" + "="*50 + "\n")

# # 2. Skriv ut den fullständiga texten för varje överlevande dokument
# print("FULLSTÄNDIG TEXT FÖR VARJE DOKUMENT:")
# for index, row in df.iterrows():
#     print(f"--- DOKUMENT ID: {row['doc_id']} ---")
#     print(f"URL: {row['url']}")
#     print("TEXT:")
#     print(row['text'])
#     print("\n" + "-"*50 + "\n")

ÖVERSIKT AV DATAN:


,doc_id,url,text
0,test_01_valid,https://test.se/valid,Artificiell intelligens och maskininlärning ha...




FULLSTÄNDIG TEXT FÖR VARJE DOKUMENT:
--- DOKUMENT ID: test_01_valid ---
URL: https://test.se/valid
TEXT:
Artificiell intelligens och maskininlärning har under det senaste årtiondet genomgått en enorm utveckling. Från att tidigare ha varit ett smalt forskningsområde, är teknologin nu en integrerad del av vår vardag. Vi ser det i allt från röstassistenter i våra mobiltelefoner till avancerade algoritmer som styr självkörande bilar och medicinsk diagnostik. Denna utveckling drivs framförallt av tillgången till enorma mängder data och kraftigare processorer som kan träna djupa neurala nätverk. Samtidigt väcker tekniken viktiga etiska frågor kring integritet, arbetsmarknadens förändring och hur vi säkerställer att AI-system fattar rättvisa och transparenta beslut i samhället.

--------------------------------------------------



In [21]:

#!pip install py-data-juicer datasets pyarrow "selectolax<1.0"

import os, subprocess, pandas as pd

# Delete the old output first so a crash can never show stale results
if os.path.exists("culturax_sv_cleaned.parquet"):
    os.remove("culturax_sv_cleaned.parquet")

res = subprocess.run(["dj-process", "--config", "config.yaml"], capture_output=True, text=True)
if res.returncode != 0 or not os.path.exists("culturax_sv_cleaned.parquet"):
    print(res.stderr[-4000:])
    raise RuntimeError("Data-Juicer failed - see the log above")

cleaned_df = pd.read_parquet("culturax_sv_cleaned.parquet")
print(f"Original documents: 10,000")
print(f"Cleaned seed documents: {len(cleaned_df):,}")
display(cleaned_df.head(3))

Original documents: 10,000
Cleaned seed documents: 7,450


,text,timestamp,url,source,doc_id
0,Med ett Minor Field Studies stipendium kan du ...,2020/04/07 07:43:23,https://www.du.se/sv/Utbildning/under-studiern...,mC4,0
1,"Kreativ marknadskommunikation, Kurs, - Luleå t...",2021/12/07 03:36:07,https://www.ltu.se/edu/course/M00/M0035N/M0035...,mC4,1
2,Den bästa idén genom tiderna | Aftonbladet\nDe...,2017/10/21 13:35:26,https://www.aftonbladet.se/nyheter/article1163...,mC4,2


In [22]:
import pandas as pd
import re

# 1. Load both datasets
raw_df = pd.read_parquet("culturax_sv_full_folder/chunk_0.parquet")
clean_df = pd.read_parquet("culturax_sv_cleaned.parquet")

# 2. Dataset-level Summary
raw_lengths = raw_df["text"].str.len()
clean_lengths = clean_df["text"].str.len()

print("=" * 60)
print(f"Dataset Size Reduction: {len(raw_df):,} -> {len(clean_df):,} "
      f"({(1 - len(clean_df)/len(raw_df))*100:.1f}% dropped)")
print(f"Original Text Length: min={raw_lengths.min()}, median={raw_lengths.median():.0f}, max={raw_lengths.max()}")
print(f"Cleaned Text Length:  min={clean_lengths.min()}, median={clean_lengths.median():.0f}, max={clean_lengths.max()}")
print("=" * 60)

# 3. Sanity Checks on Cleaned Data
html_tags_remaining = clean_df["text"].apply(lambda t: bool(re.search(r"<[^>]+>", t))).sum()
exact_duplicates = clean_df["text"].duplicated().sum()
under_min_len = (clean_lengths < 200).sum()
over_max_len = (clean_lengths > 8000).sum()

print("Integrity Checks:")
print(f" - Documents with remaining HTML tags: {html_tags_remaining}")
print(f" - Exact duplicate documents:          {exact_duplicates}")
print(f" - Documents outside length bounds:    {under_min_len + over_max_len}")
print("=" * 60)

# 4. Print 3 Cleaned Text Samples (Untruncated preview)
print("\nSample Cleaned Documents:\n")
for i, sample in enumerate(clean_df["text"].sample(3, random_state=42), 1):
    preview = sample[:400].replace("\n", " ") + ("..." if len(sample) > 400 else "")
    print(f"[{i}] ({len(sample)} chars):")
    print(f"{preview}\n")

Dataset Size Reduction: 10,000 -> 7,450 (25.5% dropped)
Original Text Length: min=106, median=1788, max=154086
Cleaned Text Length:  min=207, median=1730, max=7995
Integrity Checks:
 - Documents with remaining HTML tags: 7
 - Exact duplicate documents:          0
 - Documents outside length bounds:    0

Sample Cleaned Documents:

[1] (2191 chars):
Det finns en scen där Legolas springer på en bro som håller på att rasa och det blir bara ospännande cgi. Jämför t ex med en scen ur Furious 7 där Paul Walker springer på en buss som håller på att falla utför ett stup. Oj, så intensivt spännande det var där. Här känns insatserna alltid obefintliga. Men… … jag gillar ju sagor och fantasy så därför blir inte The Battle of the Five Armies ett bottenn...

[2] (564 chars):
Om din mor söker bostadstillägg och felaktigt anger exempelvis att ditt syskon inte är bosatt hos henne eller inte räknar in hennes inkomst, är det stor sannolikhet att din mor gör sig skyldig till bidragsbrott och riskerar böt

In [23]:
import pandas as pd
import difflib


# Load both datasets
raw_df = pd.read_parquet("culturax_sv_full_folder/chunk_0.parquet")
clean_df = pd.read_parquet("culturax_sv_cleaned.parquet")

# Merge on doc_id
merged = pd.merge(
    raw_df[["doc_id", "text", "url"]].rename(columns={"text": "raw_text"}),
    clean_df[["doc_id", "text"]].rename(columns={"text": "clean_text"}),
    on="doc_id"
)

# Filter for modified documents
modified_docs = merged[merged["raw_text"] != merged["clean_text"]]

print(f"Surviving documents: {len(merged):,}")
print(f"Documents modified in-place: {len(modified_docs):,}\n")

if not modified_docs.empty:
    # random_state removed to ensure a different sample on every run
    sample = modified_docs.sample(1).iloc[0]

    print(f"Document ID: {sample['doc_id']}")
    print(f"Source URL:  {sample['url']}\n")

    print("=== RAW (BEFORE) ===")
    print(sample["raw_text"][:1000] + "...\n")

    print("=== CLEANED (AFTER) ===")
    print(sample["clean_text"][:1000] + "...\n")

    print("=== REMOVED ARTIFACTS / DIFF ===")
    diff = difflib.ndiff(sample["raw_text"][:300].splitlines(keepends=True),
                         sample["clean_text"][:300].splitlines(keepends=True))
    for line in diff:
        if line.startswith("- ") or line.startswith("+ "):
            print(line.strip())
else:
    print("No modified documents found.")

Surviving documents: 7,450
Documents modified in-place: 2,774

Document ID: 2986
Source URL:  https://skolbanken.unikum.net/skolbanken/planering/4820763360

=== RAW (BEFORE) ===
Pedagogisk planering i Skolbanken: Måltidspedagogik
https://start.unikum.net/unikum/content/lpp/svedala/166/svedala_kraladda2/4820763360_d0945fec-29d3-4a37-bb9b-5629b2f28616.jpg
Skapad 2019-08-26 14:03 i Erlandsdals förskola Svedala unikum.net
Måltidspedagogik enligt SAPERE-metoden. SAPERE handlar om att uppleva mat med alla sinnen och att träna våra sinnen.
En del av gruppen arbetade förra terminen med måltidspedagogik enligt SAPERE-metoden. För en del barn är detta en ny upplevelse. Vi vill arbeta med detta ämne för att medvetandegöra barnen om matens betydelse för människan samt att måltiderna blir en bra och lärorik stund för alla.
Vårt mål är att öka nyfikenheten och intresset för olika mat och smakupplevelser. Vår förhoppning är att barnen på ett lustfyllt och praktiskt vis kommer vilja och våga smaka på 

In [24]:
# Export cleaned documents as seeds for synthetic text generation
import json, pandas as pd

seeds = pd.read_parquet("culturax_sv_cleaned.parquet")
with open("sv_seeds.jsonl", "w", encoding="utf-8") as f:
    for _, r in seeds.iterrows():
        f.write(json.dumps({"doc_id": r["doc_id"], "url": r["url"], "text": r["text"]}, ensure_ascii=False) + "\n")
print(f"Wrote {len(seeds):,} seed documents to sv_seeds.jsonl")

Wrote 7,450 seed documents to sv_seeds.jsonl
